<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Line Detection

Line detection involves extracting line segments from an image, focusing on specific colors that correspond to lane markers (white, yellow and red).
This process combines multiple stages color filtering, edge detection and line segment extraction, and is designed to efficiently detect lane lines in Duckietowns.

## Preprocessing

Images are preprocessed for more effective feature extraction.
For example, blurring the image using a Gaussian blur, converting the image to the HSV (Hue, Saturation and Value) color space.
HSV is preferred over RGB (Red, Green and Blue) for color segmentation, as it separates chromatic content (hue) from intensity (value), making it easier to isolate specific colors.
Then, a color range filter is applied to segment regions corresponding to marker colors (e.g., white, yellow and red).
This operation results in a binary mask, highlighting only the regions of interest.

```{todo}
Viewing the output of the color filter
```

## Edge detection

To identify edges, filtered images are passed through the [Canny edge detection algorithm](https://en.wikipedia.org/wiki/Canny_edge_detector), which consists of a gradient calculation, non-maximum suppression and hysteresis thresholding.

**Gradient calculation**: The image gradient is computed using the Sobel operator, which highlights areas of rapid intensity change ($G = \sqrt{G_x^2 + G_y^2}$, where $G_x = \frac{\partial I}{\partial x}$ and $G_y = \frac{\partial I}{\partial y}$).

**Non-maximum suppression**: Edges are thinned out by suppressing the pixels that are not part of the edge contours.

**Hysteresis thresholding**: Two thresholds ($T_{low}$ and $T_{high}$) are applied to classify edges as strong, weak or irrelevant.
Edges stronger than $T_{high}$ are retained and weak edges connected to strong ones are kept.

```{todo}
How to view the edges
```

## Line segment extraction

The detected edges are analyzed to extract line segments using the [Hough line transform](https://en.wikipedia.org/wiki/Hough_transform). This method identifies lines in the image by voting in the parameter space:

* Each edge point $(x, y)$ contributes a vote for all possible lines passing through it, represented in polar coordinates as $\rho = x\cos(\theta) + y\sin(\theta), $, where $\rho$ is the distance from the origin to the line and $\theta$ is the angle of the line normal.
* A threshold on the accumulator ensures that only lines with sufficient votes are retained.

While these are the main components involved, there are additional post processing steps that can be done, as well as the tuning of some hyperparameters to optimize performance.

```{todo}
How to view the line segments
```

## Tuning the HSV thresholds

To view the colormaps:

1. Run `dts duckiebot image_viewer DUCKIEBOT_NAME`.
2. Select the `NODE/line_detector_node/debug/maps/jpeg` topic, where `NODE` is `DUCKIEBOT_NAME/node/image_relayer`.

If the white or yellow regions of the image are not being well segmented, try tuning the color thresholds using a new `dt-core/packages/line_detector/config/line_detector_node/DUCKIEBOT_NAME.yaml` file.
The color thresholds are specified by the following thresholds, which are in HSV space as described above:

```yaml
colors:
  RED:
    low_1: [0,140,100]
    high_1: [15,255,255]
    low_2: [165,140,100]
    high_2: [180,255,255]
  WHITE:
    low: [0,0,150]
    high: [180,100,255]
  YELLOW:
    low: [15,80,50]
    high: [45,255,255]
```

**NOTE**:
You should not need to worry about the `RED` colors for now but the `WHITE` and `YELLOW` colors may need to be tuned depending on the type and amount of light in your environment.

